# 🛡️ VAHAAN — 3-Stage AI Pipeline (Timeout-Safe Edition)

## Features:
1.  **Auto-Resume**: Fetches `last.pt` from HF and continues training.
2.  **Timer Guard**: Automatically stops and saves 30 mins before the 12-hour Kaggle limit.
3.  **3-Stage Strategy**: Fine-tunes `best_new.pt` for Stage 1; trains others from COCO.

**Instructions:**
- Add your HuggingFace Token to Kaggle Secrets as `HF_TOKEN`.
- Set your `HF_REPO` below.
- Use **Accelerator: Dual T4**.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════
HF_REPO    = "YOUR_HF_USERNAME/vahaan-training-hub"
HF_TOKEN   = ""   # If not using Kaggle Secrets

TIMEOUT_HOURS = 11.5    # Save and Stop at 11 hours 30 mins
EPOCHS         = 100
IMGSZ          = 640
BATCH          = 16
# ══════════════════════════════════════════════════════════════

In [ ]:
%pip install -q ultralytics huggingface_hub pyyaml
import os, shutil, yaml, time
from pathlib import Path
from ultralytics import YOLO
from huggingface_hub import snapshot_download, HfApi
print('✅ Code base ready')

In [ ]:
DATA_ROOT   = Path('/kaggle/working/vahaan_data')
OUTPUT_DIR  = Path('/kaggle/working/vahaan_models')
DATA_ROOT.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
START_TIME  = time.time()

print(f'📥 Downloading data and base models from: {HF_REPO}')
snapshot_download(repo_id=HF_REPO, repo_type='dataset', local_dir=str(DATA_ROOT), token=HF_TOKEN or None)
print('✅ Download Complete')

In [ ]:
def sync_to_hf(file_path, repo_id, token):
    """Push a single file to HF with error handling."""
    api = HfApi()
    try:
        api.upload_file(
            path_or_fileobj=str(file_path), 
            path_in_repo=f"checkpoints/{Path(file_path).name}", 
            repo_id=repo_id, 
            repo_type='dataset', 
            token=token
        )
        print(f'   ☁️ Synced to HF: {Path(file_path).name}')
    except Exception as e:
        print(f'   ❌ HF Sync Error: {e}')

In [ ]:
# ── TRAINING LOOP WITH TIMEOUT GUARD ──

def train_with_guard(model_path, data_yaml, name, base_epochs):
    elapsed = (time.time() - START_TIME) / 3600
    if elapsed > TIMEOUT_HOURS:
        print(f'⏭️ TIMEOUT REACHED ({elapsed:.2f}h). Skipping {name}.')
        return

    print(f'\n🚀 Training {name}...')
    
    # Check for Resume
    resume_path = DATA_ROOT / 'checkpoints' / f'{name}_last.pt'
    actual_model = str(resume_path) if resume_path.exists() else str(model_path)
    if resume_path.exists(): print(f'   🔄 Resuming from checkpoint: {resume_path}')

    model = YOLO(actual_model)
    
    # Register a callback to check time every epoch
    def on_train_epoch_end(trainer):
        curr_elapsed = (time.time() - START_TIME) / 3600
        if curr_elapsed > TIMEOUT_HOURS:
            print(f'\n🚩 EMERGENCY STOP: Time limit {TIMEOUT_HOURS}h reached!')
            trainer.stop = True # Signal YOLO to stop

    model.add_callback("on_train_epoch_end", on_train_epoch_end)

    results = model.train(
        data=str(data_yaml),
        epochs=base_epochs,
        imgsz=IMGSZ,
        batch=BATCH,
        name=name,
        device=[0,1] if os.environ.get('KAGGLE_GPU_COUNT','0') == '2' else 0,
        resume=resume_path.exists()
    )

    # Save & Sync after train (or stop)
    best = Path(f'runs/detect/{name}/weights/best.pt')
    last = Path(f'runs/detect/{name}/weights/last.pt')
    
    if best.exists(): shutil.copy(best, OUTPUT_DIR / f'{name}.pt')
    if last.exists():
        local_last = OUTPUT_DIR / f'{name}_last.pt'
        shutil.copy(last, local_last)
        sync_to_hf(local_last, HF_REPO, HF_TOKEN)
    
    print(f'✅ {name} Cycle Finished.')

In [ ]:
# ── EXECUTE ──
# 1. Stage 1 (best_new.pt base)
train_with_guard(DATA_ROOT/'base_models'/'best_new.pt', DATA_ROOT/'stage1'/'data.yaml', 'stage1_vahaan', 50)

# 2. Stage 3 (COCO base)
train_with_guard('yolov8n.pt', DATA_ROOT/'stage3'/'data.yaml', 'safety_net', 100)

# 3. Pothole (COCO base)
train_with_guard('yolov8n.pt', DATA_ROOT/'pothole'/'data.yaml', 'pothole_net', 80)